# Lecture 20 — plotting with matplotlib: every figure is a `fig` and an `ax`
### EES 3350/5350 · Python in Earth Science · Wednesday, October 21, 2026 

You've already made plots: `plt.plot()` back in Lecture 06, in the loop preview, in Lab 06, and `monthly_high.plot()` on Monday. Today, we slow down and learn what's actually going on, so that you can make **any** figure look the way you want.

**Learning objectives**
- make a figure with `fig, ax = plt.subplots()` and say what the **figure** and the **axes** are
- build a figure step by step: size, labels with units, colors, a legend, a reference line, an annotation
- **zoom** in and out with `ax.set_xlim()` and `ax.set_ylim()`, and explain how that differs from slicing the data first
- choose a **line plot**, **scatter plot**, or **histogram** based on the question you're asking
- save a fiure with `fig.savefig()`

We'll keep using the Nashville 2025 daily weather from the pandas lectures.

---
## Part 0 : get todays notebook
```
cd ~/course-materials
git pull
cp week09/20_matplotlib_intro.ipynb ~/LastName_EES3350/labs/week09/20_matplotlib_intro_LastName.ipynb
```
Load the data the same way as Lectures 18 and 19:

In [ ]:
!pwd

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

raw = pd.read_csv("../week08/nashville_ghcnd_2025.csv", parse_dates=["DATE"], index_col ="DATE")
#raw = pd.read_csv("../../../course-materials/week08/nashville_ghcnd_2025.csv", parse_dates=["DATE"], index_col ="DATE")

weather = raw[["TMAX", "TMIN", "PRCP", "AWND"]].copy()
weather.head(3)

---
## Part 1 : why plot?

Here's everything `.describe()` can tell us about the daily highs:

In [ ]:
weather["TMAX"].describe()

That's useful, but it's 8 numbers summarizing 365. Now look at the **same** 365 numbers:

In [ ]:
fig, ax = plt.subplots()
ax.plot(weather.index, weather["TMAX"])
plt.show()

<div class="alert alert-warning">

**Question**: What can you see in the plot that the `.describe()` table **can't** tell you? Name 2 things.
    
</div>

<details>
<summary style="color:green;"><strong>Some answers.</strong></summary>
    
- **when** things happen: the hottest stretch is in late summer, and the coldest days are in January
- **how fast** things change: the highs can jup 20-30 $^\circ$F in a day or two in winter, but say steady in the summer
- **odd** days: the one January dip goes far below the rest (that's January 20, the 23 $^\circ$F high from Lecture 17)

`.describe()` gives you the 8 numbers. The plot gives you the story. Figures are for **finding patterns**, **checking data for mistakes**, and **showing other people** what you found.

---
## Part 2 : the figure and the axes
That cell had two lines. Let's take them apart.
```python
fig, ax = plt.subplots()
ax.plot(weather.index, weather["TMAX"])
```

`plt.subplots()` makes **two** things and hands them back as a tuple, which we unpack into two names (the same way `for i, x in enumerate(..)` unpacks a pair):
|**name**|**what it is**|**think of it as**
|:--|:--|:--|
|`fig`| the **Figure**: the whole image|the blank page|
|`ax`| the **Axes**: one plotting area, with an x-axis, a y-axis, and the data| one graph drawn on the page|

A figure can hold several axes (that's Monday, "subplots"). **Almost everything you do goes through `ax`**: drawing data, labels, limits, legends. The only thing you'll do with fig today is save it.

<img src="https://matplotlib.org/stable/_images/sphx_glr_anatomy_001.png" width="520" alt="Matplotlib's anatomy-of-a-figure diagram, labeling the figure, axes, title, x and y axis labels, ticks, tick labels, legend, grid, line, and markers">

*Every part of a figure has a name. From teh matplotlib gallerty, ["Anatomy of a figure"](https://matplotlib.org/stable/gallery/showcase/anatomy.html).*

### The same figure, two ways

Before today, you made plots with `plt.figure()` and `plt.plot()`. That still works, and you'll see it all over the internet. Here's the same figure written both ways. 

In [ ]:
plt.figure(figsize=(10,4))
plt.plot(weather.index, weather["TMAX"])
plt.xlabel("Date")
plt.ylabel("Temperature (F)")
plt.title("Nashville daily high temperature, 2025")
plt.show()

**the `ax` way** (what we'll write form now on):

In [ ]:
fig, ax = plt.subplots(figsize=(10,4))
ax.plot(weather.index, weather["TMAX"])
ax.set_xlabel("Date")
ax.set_ylabel("Temperature (F)")
ax.set_title("Nashville daily high temperature, 2025")
plt.show()

Same picture. The difference is **who you're talking to**:

- `plt.` draws on whatever figure is "current", meaning the last one made. With one graph, that's never a problem.
- `ax.` names the exact graph to draw on. On Monday, a figure will hold two or more graphs, and you'll need to say **which** one gets the title or the line.

So from here on we'll use `fig, ax`. When you find `plt.` code online, the table below can help you translate it:

| `plt` style (what you've seen) | `ax` style (from now on) |
|---|---|
| `plt.plot(x, y)` | `ax.plot(x, y)` |
| `plt.scatter(x, y)` | `ax.scatter(x, y)` |
| `plt.hist(x)` | `ax.hist(x)` |
| `plt.xlabel("...")` | `ax.set_xlabel("...")` |
| `plt.ylabel("...")` | `ax.set_ylabel("...")` |
| `plt.title("...")` | `ax.set_title("...")` |
| `plt.xlim(a, b)` | `ax.set_xlim(a, b)` |
| `plt.legend()` | `ax.legend()` |
| `plt.axhline(y)` | `ax.axhline(y)` |
| `plt.savefig("...")` | `fig.savefig("...")` |

The pattern: **drawing** has the same name: **setting** something gets a `set_` in front.

---
## Part 3 : build a figure one step at a time

We'll start with the default plot above and improve it one change at a time. Each cell is the **whole** figure again, with the new lines marked `# NEW`. (This approach is borrowed from Nicolas Rougier's [matplotlib tutorial](https://github.com/rougier/matplotlib-tutorial).)

### Step 1 : size, lavels, and a title

A figure with no labels is a figure no one else can read. Every axis lavel needs **what** it is and its **units**.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))                # NEW: (width, hight) in inches

ax.plot(weather.index, weather["TMAX"])

ax.set_xlabel("Date")                                  # NEW
ax.set_ylabel("Temperature (F)")                       # NEW
ax.set_title("Nashville daily high temperature, 2025") # NEW


<div class="alert alert-success">

### Tip: What's that `Text(0.5, 1.0, 'Nashville daily high temperature, 2025')` line above the figure?
Jupyter prints whatever the **last line** of a cell gives back. `ax.set_title()` gives back the title it made, so Jupyter prints it. (`ax.legend()` gives back `<matplotlib.legend at 0x...>`, and `ax.plot()` gives back `[<matplotlib.lines.Line2D at 0x...]`.) It's harmless, so you can ignore it, but if you hate i, put a smicolon at the end of the last line and Jupyter stays quite. Or type `plt.show()` as the last line. I tend to do the latter.
    
</div>

### Step 2 : a second line, colors, and a legend

Call `ax.plot()` again and a new line goes on the **same** axes. Give each line a `label=` and `ax.legend()` uses those labels to build the key.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))                

ax.plot(weather.index, weather["TMAX"], color="red", label="Daily high")  # NEW: color, label
ax.plot(weather.index, weather["TMIN"], color="blue", label="Daily low")  # NEW

ax.set_xlabel("Date")                                  
ax.set_ylabel("Temperature (F)")                       
ax.set_title("Nashville daily high temperature, 2025")
ax.legend();

<div class="alert alert-success">

### Tip: colors and line styles
- `color=` takes a name like `"red"`, `"black"`, `"gray"`, or one of matplotlib's default colors `"tab:blue"`, `"tab:orange"`, `"tab:green"`, `"tab:red"`, `"tab:purple"`, etc (the `tab:` colors are designed to be easy to tell apart)
- `linestyle=` takes `"-"` (solid), `"--"` (dashed), `":"` (dotted), or `"-."` (dash-dot)
- `linewidth=` takes a number; the default is 1.5

A red line next to a blue line works for most readers. Red next to **green** is hard to tell apart for about 1 in 12 men with red-green color blindness, so afoid using those two as your ownly difference.
    
</div>

### Step 3 : a reference line
`ax.axhline(y)` draws a **h**orizontal line across the whole axis at height `y`. We used this in Lab 06. Freezing is a good reference here.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))                

ax.plot(weather.index, weather["TMAX"], color="tab:red", label="Daily high")  
ax.plot(weather.index, weather["TMIN"], color="tab:blue", label="Daily low")
ax.axhline(32, color="gray", linestyle="--", label="Freezing (32 F)")            # NEW

ax.set_xlabel("Date")                                  
ax.set_ylabel("Temperature (F)")                       
ax.set_title("Nashville daily high temperature, 2025")
ax.legend();

<div class="alert alert-danger">

### Common mistake: adding a figure in a **new** cell
A figure is drawn when its cell finishes running. If you write `ax.set_title("..")` in the **next** cell, nothing new appears. Put everything for one figure in **one** cell, and rerun the whole cell when you change something.

That's why each step above repeats the whole figure.
    
</div>

<div class="alert alert-info">

### Exercise 1 : the daily average
1. Add a `TAVG` column to `weather`: the average of `TMAX` and `TMIN` (Lecture 18).
2. Make a new figure (10 by 4 inches) with `TAVG` as a line, in any color but the default.
3. Add a horizontal dashed line at the **annual mean** of `TAVG`. Don't type the number in, calculate it.
4. Label both axes (with units), add a title, and give both liens labels so the legend explains them.
    
</div>

In [ ]:
# your answer here

---
### Part 4 : zooming in and out
Look at the right edge of the Step 3 figure: the x-axis runs past the data into **2026**, and there's empty space above and below the lines. matplotlib picks the axis **limits** for you, adding a little padding. You can set them yourself:
- `ax.set_xlim(left, right)` sets the x-axis limits
- `ax.set_ylim(bottom, top)` sets the y-axis limits

### Zoom out: fix the y-axis
Setting the y-limits yourself as is useful when you want two figures to be easy to compare or to leave room for a label.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))                

ax.plot(weather.index, weather["TMAX"], color="tab:red", label="Daily high")  
ax.plot(weather.index, weather["TMIN"], color="tab:blue", label="Daily low")
ax.axhline(32, color="gray", linestyle="--", label="Freezing (32 F)")            

ax.set_ylim(0, 110)                                                           # NEW

ax.set_xlabel("Date")                                  
ax.set_ylabel("Temperature (F)")                       
ax.set_title("Nashville daily high temperature, 2025")
ax.legend();

### Zoom in: pick a window in time
The x-axis here is **dates**, so the limits have to be dates too. `pd.Timestamp("2025-07-01"` turns a string into a single date (a `Timestamp` is one entry of the DatetimeIndex you met on Wednesday).

Let's zoom in on summer:

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))                

ax.plot(weather.index, weather["TMAX"], color="tab:red", label="Daily high")  
ax.plot(weather.index, weather["TMIN"], color="tab:blue", label="Daily low")
ax.axhline(32, color="gray", linestyle="--", label="Freezing (32 F)")            

ax.set_xlim(pd.Timestamp("2025-07-01"), pd.Timestamp("2025-08-31"))          # NEW
ax.set_ylim(0, 110)

ax.set_xlabel("Date")                                  
ax.set_ylabel("Temperature (F)")                       
ax.set_title("Nashville daily high temperature, 2025")
ax.legend();

The data outside July-August is still **in** the plot; it's just outside the window we're looking through. Notice the y-axis stayed at 0-110 because we set it. That leaves a lot of empty space, and the freezing line is no help in summer.

### Zooming vs. slicing

There is a second way to "zoom": slice the **data** first with `.loc`, and plot only that. Don't set any limits, and see what matplotlib does:

In [ ]:
summer = weather.loc["2025-07-01":"2025-08-31"]

fig, ax = plt.subplots(figsize=(10, 4))

ax.plot(summer.index, summer["TMAX"], color="tab:red", label="Daily high")
ax.plot(summer.index, summer["TMIN"], color="tab:blue", label="Daily low")

ax.set_xlabel("Date")
ax.set_ylabel("Temperature (F)")
ax.set_title("Nasville daily temperature, summer 2025")
ax.legend();

Now matplotlib only sees July and August, so it picks limits that fit **just those values**: the y-axis shrinks to the summer range, and the day-to-day wiggles are much easier to see.

But look at the x-axis: the dates are frwoded and run into each other. When you zoom in on dates, matplotlib labels more of them. one lines fixes it, by tilting the labels:
```python
fig.autofmt_xdate()
```
**Add it to the end of the cell above and rerun it.**
| |**`ax.set_xlim()`/`ax.set_ylim()`**|**slice first with `.loc`**|
|:--|:--|:--|
|what changes| the **window** you look through| the **data** you plot|
|data outside the window| still there, just not visible| not plotted at all|
|other axis| stays where it was (or where you set it)| matplotlib refits it to the slice|

Neither is"right". It depends on what you want the reader to compare.

<div class="alert alert-success">

### Tip: zooming with the mouse

Everything above zoom with **code**, which is what you'll use in a saved figure. If you want to explore a plot with your mouse first, run this line once, at the top of a notebook, **before** you make any figures

```python
%matplotlib widget
```

Your figures get a small toolbar: the magnifying glass zooms to a box you draw, the four-way arrows pan, and the house button resets. Once you've found the window you like, read the limits of the axes and put them into `set_xlim()` and `set_ylim()`
    
</div>

<div class="alert alert-info">

### Exercise 2 : the January cold snap

1. Copy the Step 3 figure (highs, lows, freezing line) and **zoom in** to January 1 - February 28, 2025, with the y-axis running from 0 to 80 $^\circ$F.
2. Change the title to match.
3. Lookat your figure: on which days did the **high** stay at or below freezing? Then check yourself with a mask.
    
</div>

In [ ]:
# your code here

---
## Part 5 : point at something, then save it

### Annote
`ax.annotate()` writes text on the plot, with an optional arrow pointed at a spot:
```python
ax.annotate("the text", xy=(x_point, y_point), xytext=(x_text, y_text), arrowprops={"arrowstyle": "->"})
```
- `xy` is the spot the arrow **points to**
- `xytext` is where the **text goes**
- both are in data units, so on this figure x is a date and y is a $^\circ$F

Let's point out the hottest day of the year. We find it with `.idxmax()` instead of typing it in:

In [ ]:
hot_day = weather["TMAX"].idxmax()
hot = weather["TMAX"].max()

fig, ax = plt.subplots(figsize=(10, 4))

ax.plot(weather.index, weather["TMAX"], color="tab:red", label="Daily high")
ax.plot(weather.index, weather["TMIN"], color="tab:blue", label="Daily low")

ax.axhline(32, color="gray", linestyle="--", label="Freezing (32 F)")

ax.annotate(f"Hottest day: {hot} F on {hot_day:%B %d}",                   # NEW
            xy=(hot_day, hot),
            xytext=(pd.Timestamp("2025-03-15"), 103),
            arrowprops={"arrowstyle": "->"})

ax.set_xlim(pd.Timestamp("2025-01-01"), pd.Timestamp("2025-12-31"))
ax.set_ylim(0, 110)

ax.set_xlabel("Date")
ax.set_ylabel("Temperature (F)")
ax.set_title("Nashville daily temperatures, 2025")
ax.legend(loc="lower center")                                           # NEW: move the legend

Notice two more things in that cell:
- `ax.set_xlim()` not runs from Jan 1 to Dec 31, so the empty stretch into 2026 is gone. The `set_ylim(0,110)` from Part 4 leaves room for the label at the top
- `ax.legend(loc="lower center")` moves the legend out of the way. Other options:
    - `"upper left"`
    - `"upper right"`
    - `"lower left"`
    - `"lower right"`
    - `"center"`
    - `"best"`
        - this is the default; matplotlib will try to find an empty spot, but it doesn't always work
     
### Save

The figure lives in your notebook. To use it anywhere else (a report, a poster, slides) save it as a file. This line goes at the **end** of the same cell as the figure, or in the next cell (`fig` still remembers the figure):

In [ ]:
fig.savefig("nashville_2025_temps.png", dpi=300, bbox_inches="tight")

- the **file extension** picks the format: `.png` for slides and documents, `.pdf` for figures you'll edit or print
- `dpi=300` (dots per inch) makes it sharp enough to print. The notebook shows a lower-resolution version.
- `bbox_inches="tight"` trims extra white space and keeps labels from being cut off

The file is saved in your current working directory. So in this case it should be saved next to your notebook in `week08`. Check that it's there with the file browser on the left.

---
## Part 6 : pick the plot for the question

So far, every figure has been a **line plot**. Lines connect each point to the next one, which makes sense when the order matters, like time. Other questions need other plots.

|**the question**|**the plot**|**matplotlib**|
|:--|:--|:--|
|How does it **change** over time (or distance, or depth)?|line|`ax.plot(x, y)`|
|How are two things **related**?|scatter|`ax.scatter(x, y)`|
|What values are **common**, and how **spread out** are they?|histogram|`ax.hist(x)`|

### Scatter: how are two things related?

Do warm days also ahve warm nights? each dot is one day: the high is how far to the right, the low is how high up.

In [ ]:
fig, ax = plt.subplots(figsize=(5, 5))

ax.scatter(weather["TMAX"], weather["TMIN"], s=12, alpha=0.5)

ax.set_xlabel("Daily high (F)")
ax.set_ylabel("Daily low (F)")
ax.set_title("Nashville, 2025: each dot is one day")

- `s=` is the **size** of each dot, and `alpha=` is how **transparent** it is (0 = invisible; 1 = solid). With 365 dots pilded up, a little transparency shows you where they overlap.

The dots fall along a tight band going up and to the right: warm days really do come with warm nights. That's a **strong** relationship.

Why not `ax.plot()` here? Try it: a line would connect day 1 to day 2 to day 3, zigzagging back and forth across the plot. The order of days doesn't matter for this question, so don't connect them.

### Histogram: what values are common?

A histogram splits the range of values into **bins** and counts how many values land in each one. The height of each bar is a count.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))

ax.hist(weather["TMAX"], bins=20, color="tab:red", edgecolor="white")

ax.set_xlabel("Daily high (F)")
ax.set_ylabel("NUmber of days")
ax.set_title("How often each daily high happened, Nashville 2025")

- `bins=20` splits the range(23 to 101 $^\circ$F) into 20 equal bins. Try `bins=5` and `bins=80`: too few hides the shape, too many makes it noisy.
- `edgecolor="white"` puts a thin line between the bars so you can tell them apart.

The tallest bars are in the upper 80s and low 90s: Nashville has a **lot** of hot days. The tail to the left is winter. Notice the **x-axis is now temperature**, not date: a histogram throws away *when*, and only keeps *how often*.

<div class="alert alert-info">

### Exercise 3 : how windy is a normal day?

Make a histogram of the average daily wind speed, `AWND` (units: miles per hour). Use `bins=20`, label both axes, and add a title.

**Question:** what range of wind speeds is most common? Does the histogram agree with `weather["AWND"].describe()`?
    
</div>

In [ ]:
# your answer here

<div class="alert alert-info">

### Exercise 4 : are hot days calmer?

Earlier, we found that days at 90 $^\circ$F or above averaged **4.6 mph** of wind, compared with **6.6 mph** for the whole year. An average can hide a lot, so let's look at every day.

1. What kind of plot answers "how are temperature and wind related? Make it, with the daily high on the x-axis and the wind speed on the y-axis.
2. Add a dashed vertical line at 90 $^\circ$. (Hint: if a **h**orizontal line is `axline`, what is a **v**ertical one?)
3. Look at your figure. Are hot days calmer? Is it a strong relationship, like highs vs lows, or a weak one?
    
</div>

In [ ]:
# your answer here

---
## Summary

| to... | write |
|---|---|
| make a figure and one set of axes | `fig, ax = plt.subplots(figsize=(w, h))` |
| draw a line (things that change in order) | `ax.plot(x, y, color=..., linestyle=..., label=...)` |
| draw dots (how two things relate) | `ax.scatter(x, y, s=..., alpha=...)` |
| count values in bins (what's common) | `ax.hist(x, bins=...)` |
| label axes and title | `ax.set_xlabel()`, `ax.set_ylabel()`, `ax.set_title()` |
| show a key for the `label=`s | `ax.legend(loc=...)` |
| reference lines | `ax.axhline(y)`, `ax.axvline(x)` |
| zoom | `ax.set_xlim(left, right)`, `ax.set_ylim(bottom, top)` |
| a single date (for date axes) | `pd.Timestamp("2025-07-01")` |
| point at something | `ax.annotate("text", xy=(...), xytext=(...), arrowprops={"arrowstyle": "->"})` |
| tilt crowded date labels | `fig.autofmt_xdate()` |
| save | `fig.savefig("name.png", dpi=300, bbox_inches="tight")` |

**On Monday:** more than one axes in a figure (subplots), what makes a figure *good*, and then Lab 09, a tour of the plot types you can make with data from the whole semester.

### Further reading
- Jake VanderPlas, [*Python Data Science Handbook*, Chapter 4: Visualization with Matplotlib](https://jakevdp.github.io/PythonDataScienceHandbook/04.00-introduction-to-matplotlib.html). Sections 4.01 (line plots), 4.02 (scatter plots), and 4.05 (histograms) match today.
- Nicolas P. Rougier, [Matplotlib tutorial](https://github.com/rougier/matplotlib-tutorial) (CC BY-SA 4.0). The step-by-step build in Part 3 follows his approach.
- [matplotlib gallery](https://matplotlib.org/stable/gallery/index.html): find a figure that looks like what you want, and copy its code.

<div class="alert alert-custom" style="background-color:#e8daef; border:1px solid #d2b4de; color:#6c3483; padding:10px; border-radius:4px;">

### Bonus : shade the daily range
`ax.fill_between(x, y1, y2)` fills the space between two lines.

1. Remake the Step 3 figure, but instead of two lines, shade the band between `TMIN` and `TMAX` with `fill_between`. Use `alpha=0.4` and give it a `label=`.
2. On top of the band, plot `TAVG` as a thin black line.
3. Zoom to the year (no 2026), set the y-limits to 0–110, and save it as a PDF.

</div>

In [ ]:
# your answer here

<details>
<summary style="color:green;"><strong>Answer.</strong></summary>

```python
fig, ax = plt.subplots(figsize=(10, 4))

ax.fill_between(wx.index, wx["TMIN"], wx["TMAX"], color="tab:orange", alpha=0.4, label="Daily low to high")
ax.plot(wx.index, wx["TAVG"], color="black", linewidth=0.8, label="Daily average")

ax.set_xlim(pd.Timestamp("2025-01-01"), pd.Timestamp("2025-12-31"))
ax.set_ylim(0, 110)

ax.set_xlabel("Date")
ax.set_ylabel("Temperature (°F)")
ax.set_title("Nashville daily temperature range, 2025")
ax.legend(loc="lower center")

fig.savefig("nashville_2025_range.pdf", bbox_inches="tight")
```
---
![answer figure](attachment:ans4_0.png)

The width of the band at each date is that day's range, `TMAX - TMIN`.

</details>

<div class="alert alert-custom" style="background-color:#e8daef; border:1px solid #d2b4de; color:#6c3483; padding:10px; border-radius:4px;">

### Bonus: what `.plot()` was doing on Monday
In Lecture 19 you wrote `monthly_high.plot()`. pandas `.plot()` is matplotlib underneath: it makes a figure and axes for you, draws, and adds some labels. It also **gives you back** the `ax`, so you can keep going:

```python
monthly_high = wx.groupby(wx.index.month)["TMAX"].mean()

ax = monthly_high.plot(marker="o", color="tab:red")
ax.set_xlabel("Month")
ax.set_ylabel("Average daily high (°F)")
ax.set_title("Nashville monthly average high, 2025")
```

Or make your own `fig, ax` first, and tell pandas which axes to draw on with `monthly_high.plot(ax=ax)`. Try it below.

</div>

In [ ]:
# your answer here